# 第 7 章:生成 —— logits 怎么变成文本

前 6 章我们追踪了「token id → embedding → 8 个 Block → lm_head → logits」这条**编码流水线**。但你有没有想过:模型吐出来的 `logits` 是一个 `(batch, seq_len, vocab_size)` 的浮点矩阵 —— **它怎么变成人类能读的文字?**

这就是**生成(generation)**要解决的问题。训练时,模型只需要做一次 forward pass 算 loss;但生成时,模型需要**一次生成一个 token**,循环往复,直到遇到结束符或达到长度上限。

本章对应 `model/model_minimind.py` 的 **256-288 行** —— 那个手写的 `generate` 方法。别被 HF 的 `GenerationMixin` 吓到,minimind **重写了它**,代码只有 32 行,却能支持 temperature、top-k、top-p、repetition penalty、KV cache、streamer 等全部现代生成特性。

> 生成是 LLM 的「输出边界」—— logits 是概率空间里的数,生成策略决定了如何从概率空间采样到离散的 token id。理解它,你才能控制模型「说什么」和「怎么说」。

我们将遵循 **naive → compact** 的路径:先写一个 15 行的朴素 greedy 循环(无 KV cache),再逐步加入 temperature、top-k、top-p、repetition penalty,最后展示 minimind 的完整 `generate`。

## 7.1 自回归生成本质:一次一个 token

LLM 生成的核心特征是**自回归(autoregressive)**:每一步的输出依赖于之前所有步骤的输出。

### 生成循环的 5 步

```
给定 prompt = [t0, t1, t2]

循环:
  1. forward(prompt)        → logits (batch, seq_len, vocab)
  2. 取最后一个位置           → logits[:, -1, :]  (batch, vocab)
  3. 选 token(某种策略)      → next_token (batch, 1)
  4. 拼回 prompt              → prompt = [t0, t1, t2, t3]
  5. 重复 1-4
```

关键洞察:**每次只取最后一个位置的 logits**。因为 Transformer 的 causal mask 保证了「位置 T 的输出只依赖位置 0 到 T 的输入」。前面位置的 logits 在生成时是**废值** —— 它们对应的「正确答案」已经知道了(就是 prompt 本身),只有最后一个位置才是「下一个待预测 token」。

> 这也是为什么自回归生成是 $O(T^2)$ 的:生成第 $t$ 个 token 要把前 $t-1$ 个 token 全部 forward 一遍。KV cache(7.7 节)就是来解决这个问题的。

### 朴素图示

```
Step 0:  [A B C] ─forward─→ logits ─→ 取最后 ─→ 选 D
Step 1:  [A B C D] ─forward─→ logits ─→ 取最后 ─→ 选 E
Step 2:  [A B C D E] ─forward─→ logits ─→ 取最后 ─→ 选 F
...
```

每一步都把**整个序列**重新喂给模型。这在正确性上没问题,但第 2 步对 A B C D 的计算和第 1 步完全重复。这正是 7.7 节 KV cache 要消灭的冗余。

现在,让我们从最简单的策略开始:**greedy decoding**。

&nbsp;

---

## 7.2 Greedy Decoding(朴素版,无 KV cache)

**贪心解码(greedy decoding)**是最简单的生成策略:每一步都选 logits 最大的那个 token。

$$\text{next\_token} = \arg\max_i \text{logits}[i]$$

数学上:`argmax` 等价于「概率最大的那个」,因为 softmax 是单调函数(logit 大 → 概率大)。所以 greedy 就是「永远选模型最有把握的词」。

代码极简,不需要任何概率采样:

In [ ]:
# 朴素 greedy decoding —— 完全手写,不用任何 HF 接口
# 假设 model 和 tokenizer 已加载

import torch

def greedy_generate_naive(model, input_ids, max_new_tokens=20, eos_token_id=2):
    """最朴素的生成循环:无 KV cache,每步重新 forward 整个序列。"""
    generated = input_ids.clone()  # 不修改原始输入
    for step in range(max_new_tokens):
        # 1. forward 整个序列
        with torch.inference_mode():
            outputs = model(generated)
        # 2. 取最后一个位置的 logits
        next_logits = outputs.logits[:, -1, :]          # (batch, vocab)
        # 3. greedy: argmax
        next_token = torch.argmax(next_logits, dim=-1, keepdim=True)  # (batch, 1)
        # 4. 拼回
        generated = torch.cat([generated, next_token], dim=-1)
        # 5. 遇到 EOS 就停
        if next_token.item() == eos_token_id:
            print(f"  step {step}: 生成 EOS,停止。")
            break
        print(f"  step {step}: → token_id={next_token.item()}")
    return generated

# 模拟演示(不需要真模型,用随机 logits 展示循环逻辑)
print("=== Greedy Decoding 模拟(随机 logits)===")
print("假设词表大小=10,模拟 5 步生成:\n")

torch.manual_seed(42)
fake_input = torch.tensor([[1, 2, 3]])  # 假装是 token ids

# 模拟 model.forward 返回随机 logits
class FakeModel:
    def __call__(self, ids):
        logits = torch.randn(ids.shape[0], ids.shape[1], 10)  # vocab=10
        class Out: pass
        out = Out(); out.logits = logits
        return out
    inference_mode = staticmethod(torch.inference_mode)

result = greedy_generate_naive(FakeModel(), fake_input, max_new_tokens=5, eos_token_id=999)
print(f"\n最终序列: {result[0].tolist()}")
print(f"输入长度 3 → 输出长度 {result.shape[1]}(生成了 {result.shape[1]-3} 个 token)")

这个循环完全正确,但有三个问题:

**问题 1:确定性导致重复**

Greedy 永远选概率最高的词。如果模型在某个位置陷入循环(比如反复输出「的的的...」),greedy 无法自救 —— 因为「的」永远是概率最高的,每一步都会选它。

**问题 2:无 KV cache,效率极低**

第 $t$ 步要 forward 前 $t-1$ 个 token,总计算量 $= 1 + 2 + ... + T = O(T^2)$。生成 100 个 token 相当于做了约 5000 次 token-level 的注意力计算。

**问题 3:缺乏多样性**

同一个 prompt 永远生成同一个输出。对聊天场景不友好 —— 用户不希望每次问「你好」都得到一模一样的回复。

> minimind 的 `generate` 默认 `do_sample=True`(不是 greedy!),正是为了避免这些问题。

接下来我们逐一解决:temperature 解决多样性,top-k/top-p 解决质量问题,repetition penalty 解决重复,KV cache 解决效率。

&nbsp;

---

## 7.3 Temperature:控制概率分布的「锐度」

**温度(temperature)**是一个标量 $T > 0$,作用于 logits 之后、softmax 之前:

$$p_i = \frac{\exp(\text{logit}_i / T)}{\sum_j \exp(\text{logit}_j / T)}$$

直觉上:

| $T$ 的值 | 行为 | 效果 |
|---|---|---|
| $T \to 0$ | 分母 $\to \exp(\max/T)$,概率集中到最大值 | **等价 greedy** |
| $T = 1$ | 标准 softmax | 原始概率分布 |
| $T > 1$ | 概率被「摊平」 | 更随机、更有创意 |

**关键数学性质**:当 $T \to 0$ 时,$\exp(\text{logit}_i / T)$ 中最大的那个趋向无穷大,其余的趋向 0 —— 所以 softmax 输出变成 one-hot,argmax 取到最大值,即 greedy。

minimind 的实现就在第 267 行:

```python
logits = outputs.logits[:, -1, :] / temperature   # 第 267 行
```

注意:先除以 temperature,**再**做后续的 top-k/top-p 和采样。

In [ ]:
# 用 matplotlib 可视化 temperature 对概率分布的影响
import matplotlib
matplotlib.use('Agg')  # 无头环境
import matplotlib.pyplot as plt
import numpy as np

# 模拟一组 logits(词表大小=20)
np.random.seed(7)
raw_logits = np.random.randn(20) * 3
raw_logits[5] = 5.0   # 人为制造一个「最优」token
raw_logits[12] = 4.2  # 次优

def softmax(x):
    e = np.exp(x - x.max())  # 数值稳定
    return e / e.sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
temperatures = [0.1, 0.85, 2.0]
titles = ['T=0.1 (≈greedy)', 'T=0.85 (minimind默认)', 'T=2.0 (混乱)']

for ax, T, title in zip(axes, temperatures, titles):
    probs = softmax(raw_logits / T)
    colors = ['#e74c3c' if i == 5 else '#3498db' if i == 12 else '#95a5a6' for i in range(20)]
    ax.bar(range(20), probs, color=colors)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('token id')
    ax.set_ylabel('probability')
    top_p = probs[5]
    ax.annotate(f'p={top_p:.3f}', xy=(5, top_p), fontsize=9, ha='center', va='bottom')

plt.tight_layout()
plt.savefig('temperature_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("图已保存:temperature_effect.png")
print(f"\nT=0.1:  token 5 的概率 = {softmax(raw_logits/0.1)[5]:.4f}  (几乎独占)")
print(f"T=0.85: token 5 的概率 = {softmax(raw_logits/0.85)[5]:.4f}  (主导但有竞争)")
print(f"T=2.0:  token 5 的概率 = {softmax(raw_logits/2.0)[5]:.4f}  (几乎被拉平)")
print(f"\n→ T 越小,分布越尖锐(确定性);T 越大,分布越平坦(多样性)")

从图中可以清楚看到:

- **T=0.1**:分布几乎是 one-hot,token 5 独占 ~99% 概率 —— 等价 greedy
- **T=0.85**(minimind 默认):token 5 仍占主导,但其他 token 有可被采样的概率 —— **兼顾确定性和多样性**
- **T=2.0**:分布几乎均匀,随机采样几乎等价于瞎选 —— **输出不可控**

> minimind 选择 `temperature=0.85` 是经验值。太低(如 0.1)会退化成 greedy,失去多样性;太高(如 1.5+)会导致胡言乱语。0.7-0.9 是大多数 LLM 的甜点区间。

但光有 temperature 还不够。即使 T=0.85,词表末尾那些概率极低的 token(比如乱码字符)偶尔也会被采到。我们需要**截断**。

&nbsp;

---

## 7.4 Top-k 采样:只保留概率最高的 k 个

**Top-k** 的思路简单粗暴:排序后只保留概率最高的 $k$ 个 token,其余的全部设为 $-\infty$(softmax 后概率为 0)。

```
原始 logits:    [3.2, 0.1, -1.0, 4.5, 2.1, 0.8, ...]  (6400 个)
top_k=3 排序后:  [4.5, 3.2, 2.1, 其余全部 → -inf]
```

minimind 的实现(第 271-272 行)极其精炼:

```python
if top_k > 0:
    logits[logits < torch.topk(logits, top_k)[0][..., -1, None]] = -float('inf')
```

拆解这行:
1. `torch.topk(logits, top_k)` 返回 top-k 的值和索引
2. `[0][..., -1, None]` 取第 k 大的那个值(即 top-k 的**下界阈值**)
3. `logits < 阈值` 的位置全部设为 $-\infty$

In [ ]:
# 手动实现 top-k filter,逐步追踪
import torch

def top_k_filter(logits, k):
    """保留概率最高的 k 个,其余设为 -inf。"""
    if k <= 0:
        return logits
    # 第 1 步:找到第 k 大的值(阈值)
    topk_values = torch.topk(logits, k, dim=-1).values   # 降序排列的前 k 个
    threshold = topk_values[..., -1, None]                # 最后一个 = 第 k 大 = 阈值
    print(f"  top-{k} 阈值(第 {k} 大的 logit): {threshold.item():.4f}")
    # 第 2 步:小于阈值的全部 -inf
    filtered = logits.clone()
    filtered[filtered < threshold] = -float('inf')
    # 统计保留了多少
    n_kept = (filtered > -float('inf')).sum().item()
    print(f"  保留 token 数: {n_kept} / {logits.shape[-1]}")
    return filtered

# 演示
torch.manual_seed(42)
fake_logits = torch.randn(1, 6400) * 3  # 模拟 minimind 的 vocab=6400

print("=== Top-k 过滤演示(vocab=6400)===")
for k in [5, 50, 200, 1000]:
    print(f"\ntop_k={k}:")
    _ = top_k_filter(fake_logits[0], k)

print("\n" + "=" * 50)
print("minimind 默认 top_k=50:从 6400 个候选中砍到 50 个。")
print("但 minimind 词表只有 6400,50/6400 = 0.78% —— 截断非常激进。")
print("对比 GPT-2(vocab=50257):50/50257 = 0.10%,合理得多。")

### minimind 为什么默认 `top_k=50` 但实际「不太管用」?

注意 minimind 的 `generate` 签名里 `top_k=50`,但这只在词表大时才有意义。minimind 的 `vocab_size=6400`(见第 2 章),top_k=50 只保留前 0.78% 的 token —— 这对大词表是合理的截断,但对小词表已经太激进了。

实际上,minimind 的生成效果主要靠 **top_p=0.85** 控制(下一节),top_k 更多是「保险阀」—— 防止极端情况下概率分散到太多 token。

> **实践建议**:词表小时(如 minimind 的 6400),top_k 设大一点(如 200)或设 0(禁用),主要靠 top_p 控制质量。

&nbsp;

---

## 7.5 Top-p(Nucleus Sampling):动态截断

Top-k 的问题在于 $k$ 是**固定的**:不管概率分布是尖锐还是平坦,都只保留 $k$ 个。但有时分布很尖锐(前 3 个就占了 99%),有时很平坦(前 50 个才占 80%)—— 固定的 $k$ 不够灵活。

**Top-p(nucleus sampling)**用概率**累积**来动态决定保留多少:

1. 按 prob 降序排列
2. 从前往后累加概率
3. 累积和**首次超过 $p$** 时,丢弃之后的所有 token

```
排序后概率:  [0.45, 0.30, 0.12, 0.08, 0.03, 0.01, ...]
累积概率:    [0.45, 0.75, 0.87, 0.95, 0.98, 0.99, ...]
top_p=0.85:   ✓     ✓     ✗(0.87>0.85,从这里开始丢弃)
→ 保留 [0.45, 0.30],丢弃其余
```

minimind 的实现(第 273-277 行):

```python
if top_p < 1.0:
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    mask = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1) > top_p
    mask[..., 1:], mask[..., 0] = mask[..., :-1].clone(), 0   # 右移一位,保留刚好超阈值的
    logits[mask.scatter(1, sorted_indices, mask)] = -float('inf')
```

In [ ]:
# 手动实现 top-p filter,逐步追踪累积概率
import torch

def top_p_filter(logits, p):
    """保留累积概率达到 p 的最小 token 集合(nucleus)。"""
    if p >= 1.0:
        return logits, None
    
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # 找到累积概率超过 p 的位置
    # 关键:右移一位,保留「刚好使累积超过 p」的那个 token
    sorted_mask = cumulative_probs > p
    sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
    sorted_mask[..., 0] = 0
    
    # 把 mask 从「排序后空间」散回「原始空间」
    mask = sorted_mask.scatter(1, sorted_indices, sorted_mask)
    
    filtered = logits.clone()
    filtered[mask] = -float('inf')
    return filtered, (sorted_indices, sorted_probs, cumulative_probs, sorted_mask)

# 演示:追踪 top_p 的累积过程
torch.manual_seed(42)
fake_logits = torch.randn(1, 20) * 4  # vocab=20 方便展示

filtered, debug = top_p_filter(fake_logits[0], p=0.85)
sorted_indices, sorted_probs, cumprobs, sorted_mask = debug

print("=== Top-p=0.85 累积过程(vocab=20)===")
print(f"{'排名':>4s} {'原始idx':>7s} {'概率':>8s} {'累积':>8s} {'保留?':>6s}")
print("-" * 38)
for rank in range(20):
    idx = sorted_indices[rank].item()
    prob = sorted_probs[rank].item()
    cum = cumprobs[rank].item()
    keep = "✗" if sorted_mask[rank].item() else "✓"
    print(f"{rank+1:>4} {idx:>7} {prob:>8.4f} {cum:>8.4f} {keep:>6}")
    if sorted_mask[rank].item() and rank < 10:
        pass  # 第一被丢弃的

n_kept = (filtered > -float('inf')).sum().item()
print(f"\n保留 {n_kept} 个 token,丢弃 {20 - n_kept} 个")
print(f"动态截断:不像 top_k 固定数量,而是固定概率质量")

### Top-k vs Top-p 对比

| 维度 | Top-k | Top-p (nucleus) |
|---|---|---|
| 截断方式 | 固定数量 | 固定概率质量 |
| 分布尖锐时 | 可能保留太多无意义 token | 自动减少保留数 |
| 分布平坦时 | 可能截断过多 | 自动增加保留数 |
| 参数直觉 | 「最多选 k 个」 | 「概率质量到 p% 就停」 |
| minimind 默认 | `top_k=50` | `top_p=0.85` |

**实际应用中,两者通常同时开启**。minimind 的 `generate` 先执行 top_k(第 271 行),再执行 top_p(第 273 行)—— 先砍数量,再砍概率尾部,双重保险。

> 经验法则:`top_p=0.9` 和 `top_p=0.85` 是 LLM 社区的主流选择。OpenAI API 默认 `top_p=1.0`(只用 temperature),而 minimind/HuggingFace 默认 `top_p=0.85`。

&nbsp;

---

## 7.6 Repetition Penalty:惩罚已出现的 token

即使有 temperature 和 top-p,模型有时仍会陷入重复 —— 尤其是生成长文本时,反复输出同一个短语。

**Repetition penalty(重复惩罚)**的思路:对**已经出现在序列中的 token**,降低它们的 logit。

$$\text{logit}_i' = \begin{cases} \text{logit}_i / \text{penalty} & \text{if } \text{logit}_i > 0 \\ \text{logit}_i \times \text{penalty} & \text{if } \text{logit}_i \leq 0 \end{cases}$$

注意:不是统一除以 penalty,而是**正负分开处理**。这是为了确保惩罚是**单调的** —— 无论原 logit 正负,惩罚后概率一定降低。

minimind 的实现(第 268-270 行):

```python
if repetition_penalty != 1.0:
    for i in range(input_ids.shape[0]):
        seen = torch.unique(input_ids[i])
        score = logits[i, seen]
        logits[i, seen] = torch.where(score > 0, score / repetition_penalty, score * repetition_penalty)
```

`penalty=1.0` 时除以/乘以 1 都不变 —— 即「不惩罚」,这也是 minimind 的默认值。

In [ ]:
# 实现 repetition penalty 并可视化效果
import torch

def apply_repetition_penalty(logits, input_ids, penalty):
    """对已出现的 token 降低 logit。"""
    if penalty == 1.0:
        return logits
    for i in range(logits.shape[0]):
        seen = torch.unique(input_ids[i])
        score = logits[i, seen]
        logits[i, seen] = torch.where(
            score > 0,
            score / penalty,
            score * penalty
        )
    return logits

# 模拟:假设 token [5, 12, 3] 已经出现
torch.manual_seed(0)
logits = torch.randn(1, 10) * 3
input_ids = torch.tensor([[5, 12, 3, 5, 12]])  # 有重复的 token

print("=== Repetition Penalty 效果对比 ===\n")
print(f"已出现 token: {torch.unique(input_ids[0]).tolist()}")

for penalty in [1.0, 1.1, 1.3, 2.0]:
    orig = logits.clone()
    modified = apply_repetition_penalty(orig.clone(), input_ids, penalty)
    
    # 看看 token 5(已出现)和 token 7(未出现)的变化
    t5_before = logits[0, 5].item()
    t5_after = modified[0, 5].item()
    t7_before = logits[0, 7].item()
    t7_after = modified[0, 7].item()
    
    print(f"\npenalty={penalty}:")
    print(f"  token 5(已出现): {t5_before:+.3f} → {t5_after:+.3f}  {'↓ 惩罚' if abs(t5_after) < abs(t5_before) else '(正数被除,缩小)'}")
    print(f"  token 7(未出现): {t7_before:+.3f} → {t7_after:+.3f}  (不变)")

print("\n" + "=" * 50)
print("penalty=1.0: 不惩罚(默认)")
print("penalty=1.1-1.3: 温和惩罚,减少重复但不生硬")
print("penalty>1.5: 过度惩罚,可能导致模型回避正常用词")

### 为什么正负分开处理?

如果统一「除以 penalty」:
- 正 logit(如 5.0)→ 5.0/1.3 = 3.85 → 概率降低 ✓
- 负 logit(如 -5.0)→ -5.0/1.3 = -3.85 → **概率反而升高** ✗(更接近 0)

所以用 `torch.where`:正数除以 penalty(缩小),负数乘以 penalty(更负),**两个方向都让概率降低**。

> 这是 HuggingFace `RepetitionPenaltyLogitsProcessor` 的标准实现。minimind 把它内联进了 `generate`,省去了 processor 机制的开销。

&nbsp;

---

## 7.7 KV Cache:核心优化 $O(T^2) \to O(T)$

到目前为止,我们的朴素生成循环每一步都 forward **整个序列**。但 Transformer 的注意力机制有一个性质:**历史 token 的 K 和 V 向量不会变**。

### 朴素 vs KV cache

```
朴素(无 cache):
  Step 0:  forward([A B C])         → 算 A B C 的 K V Q
  Step 1:  forward([A B C D])       → 重算 A B C D 的 K V Q  ← A B C 重复计算!
  Step 2:  forward([A B C D E])     → 重算 A B C D E 的 K V Q  ← A B C D 重复!

有 KV cache:
  Step 0:  forward([A B C])         → 算 K V,缓存 K₀V₀ K₁V₁ K₂V₂
  Step 1:  forward([D]) + cache     → 只算 D 的 K₃V₃,和缓存拼接
  Step 2:  forward([E]) + cache     → 只算 E 的 K₄V₄,和缓存拼接
```

每一步只 forward **1 个新 token**,用缓存的历史 K/V 做注意力。计算量从 $O(T^2)$ 降到 $O(T)$ —— **生成 1000 个 token 快几十倍**。

### Shape 追踪

minimind 的 `past_key_values` 是一个 list,每层一个 tuple `(K, V)`:

```
past_key_values = [
    (K_layer0, V_layer0),   # 每个 shape = (batch, n_heads, past_len, head_dim)
    (K_layer1, V_layer1),
    ...
    (K_layer7, V_layer7),   # 8 层
]
```

在 `generate` 循环中(第 264-265 行):

```python
past_len = past_key_values[0][0].shape[1] if past_key_values else 0
outputs = self.forward(input_ids[:, past_len:], ...)  # 只喂新 token!
```

`input_ids[:, past_len:]` 切片跳过已缓存的 token,只把**新增的部分**喂给模型。

In [ ]:
# 模拟 KV cache 的 shape 变化过程
import torch

print("=== KV Cache Shape 追踪(模拟 3 层, 生成 4 个 token)===")
print()

# 模拟参数
batch = 1
n_heads = 8
head_dim = 96
n_layers = 3
prompt_len = 3  # 初始 prompt 长度

past_key_values = None  # 初始无缓存

print(f"初始:prompt 长度={prompt_len}, past_key_values=None")
print()

for step in range(4):
    # 模拟第 264 行:计算 past_len
    if past_key_values is not None:
        past_len = past_key_values[0][0].shape[1]
    else:
        past_len = 0
    
    # 模拟第 265 行:只 forward 新 token
    new_input_len = prompt_len if step == 0 else 1
    total_len = past_len + new_input_len
    
    print(f"Step {step}:")
    print(f"  past_len = {past_len}")
    print(f"  forward input_ids[:, {past_len}:] → {new_input_len} 个新 token")
    print(f"  总序列长度 = {total_len}")
    
    # 模拟每层生成新的 K V 并和缓存拼接
    new_past = []
    for layer in range(n_layers):
        new_K = torch.randn(batch, n_heads, new_input_len, head_dim)
        new_V = torch.randn(batch, n_heads, new_input_len, head_dim)
        if past_key_values is not None:
            K = torch.cat([past_key_values[layer][0], new_K], dim=1)  # 沿 seq 维拼接
            V = torch.cat([past_key_values[layer][1], new_V], dim=1)
        else:
            K, V = new_K, new_V
        new_past.append((K, V))
    
    past_key_values = new_past
    
    # 打印 layer 0 的 shape
    print(f"  layer 0 K shape: ({batch}, {n_heads}, {total_len}, {head_dim})")
    print(f"  → 缓存了 {total_len} 个位置 × {n_layers} 层")
    print()

print("=" * 50)
print(f"朴素方案:4 步共 forward {3+4+5+6} = 18 个 token 位置")
print(f"KV cache:4 步共 forward {3+1+1+1} = 6 个 token 位置")
print(f"节省: {(1-6/18)*100:.0f}%")
print(f"\n生成 T 个 token:")
print(f"  朴素:  1+2+...+(T+3) ≈ O(T²)")
print(f"  cache: 3+1+1+...+1  ≈ O(T)")

&nbsp;

---

## 7.8 完整 generate:32 行拆解

现在把所有零件拼起来。以下是 minimind 的 `generate` 方法(`model_minimind.py:256-288`),逐段拆解:

### 第 1 段:初始化(257-262 行)

```python
@torch.inference_mode()                           # 关闭梯度,省内存
def generate(self, inputs=None, attention_mask=None, max_new_tokens=8192,
             temperature=0.85, top_p=0.85, top_k=50, eos_token_id=2,
             streamer=None, use_cache=True, num_return_sequences=1,
             do_sample=True, repetition_penalty=1.0, **kwargs):
    input_ids = kwargs.pop("input_ids", inputs).repeat(num_return_sequences, 1)
    attention_mask = attention_mask.repeat(num_return_sequences, 1) if attention_mask is not None else None
    past_key_values = kwargs.pop("past_key_values", None)
    finished = torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
    if streamer: streamer.put(input_ids.cpu())
```

- `@torch.inference_mode()`:推理模式,不记录计算图
- `.repeat(num_return_sequences, 1)`:支持批量生成(同一 prompt 生成多条)
- `finished`:追踪哪些序列已结束(EOS)
- `streamer.put`:流式输出的起始钩子

### 第 2 段:主循环 + KV cache 切片(263-266 行)

```python
for _ in range(max_new_tokens):
    past_len = past_key_values[0][0].shape[1] if past_key_values else 0
    outputs = self.forward(input_ids[:, past_len:], attention_mask, past_key_values, use_cache=use_cache, **kwargs)
    attention_mask = torch.cat([attention_mask, attention_mask.new_ones(...)], -1) if attention_mask is not None else None
```

核心是 `input_ids[:, past_len:]` —— **只 forward 新 token**。每步结束后 attention_mask 增长 1 位。

### 第 3 段:采样策略(267-278 行)

```python
logits = outputs.logits[:, -1, :] / temperature          # temperature
if repetition_penalty != 1.0: ...                        # repetition penalty
if top_k > 0: ...                                        # top-k filter
if top_p < 1.0: ...                                      # top-p filter
next_token = torch.multinomial(softmax(logits), 1) if do_sample else torch.argmax(logits, dim=-1, keepdim=True)
```

**执行顺序**:temperature → repetition penalty → top_k → top_p → sample。注意 `do_sample=False` 时退化为 greedy(argmax)。

### 第 4 段:EOS 处理 + 拼接 + 缓存更新(279-285 行)

```python
if eos_token_id is not None:
    next_token = torch.where(finished.unsqueeze(-1), next_token.new_full((..., 1), eos_token_id), next_token)
input_ids = torch.cat([input_ids, next_token], dim=-1)
past_key_values = outputs.past_key_values if use_cache else None
if eos_token_id is not None:
    finished |= next_token.squeeze(-1).eq(eos_token_id)
    if finished.all(): break
```

- 已结束的序列,强制输出 EOS(防止继续生成垃圾)
- `finished.all()`:所有序列都结束了才 break

### 第 5 段:收尾(286-288 行)

```python
if streamer: streamer.end()
if kwargs.get("return_kv"): return {'generated_ids': input_ids, 'past_kv': past_key_values}
return input_ids
```

- `streamer.end()`:流式输出结束钩子
- `return_kv`:可选返回 KV cache(用于续写场景)

### 覆盖 GenerationMixin

注意第 234 行:

```python
class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
```

minimind 继承了 `GenerationMixin`(HF 的生成基类),但**重写了 `generate`**。这意味着 HF 的 `model.generate(...)` 调用的不是 HF 默认实现,而是这 32 行手写代码。

> 为什么重写?HF 的 `GenerationMixin` 依赖 `LogitsProcessor` 和 `LogitsProcessorList` 机制,代码量大、调度复杂。手写版本更直观、可控,且去掉了不必要的抽象层。这是 minimind「最小化」哲学的体现。参见 [discussion #611](https://github.com/jingyaogong/minimind/discussions/611)。

In [ ]:
# 把 minimind 的 generate 核心逻辑浓缩成可运行的伪代码
# 这个版本去掉了 batch/streamer 等工程细节,只保留核心生成逻辑

import torch

def minimind_generate_essence(model, input_ids, max_new_tokens=20, 
                               temperature=0.85, top_p=0.85, top_k=50,
                               eos_token_id=2, repetition_penalty=1.0,
                               do_sample=True, use_cache=True):
    """minimind generate 的核心逻辑精华版。"""
    input_ids = input_ids.clone()
    past_key_values = None
    
    print(f"输入: {input_ids[0].tolist()}")
    print(f"参数: T={temperature}, top_p={top_p}, top_k={top_k}, sample={do_sample}")
    print(f"生成:\n")
    
    for step in range(max_new_tokens):
        # --- KV cache 切片 ---
        past_len = past_key_values[0][0].shape[1] if past_key_values else 0
        only_new = input_ids[:, past_len:]
        
        with torch.inference_mode():
            outputs = model(only_new, past_key_values=past_key_values, use_cache=use_cache)
        
        logits = outputs.logits[:, -1, :] / temperature       # ① temperature
        
        if repetition_penalty != 1.0:                          # ② repetition penalty
            seen = torch.unique(input_ids[0])
            score = logits[0, seen]
            logits[0, seen] = torch.where(score > 0, score / repetition_penalty, score * repetition_penalty)
        
        if top_k > 0:                                           # ③ top-k
            logits[logits < torch.topk(logits, top_k)[0][..., -1, None]] = -float('inf')
        
        if top_p < 1.0:                                         # ④ top-p
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            mask = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1) > top_p
            mask[..., 1:], mask[..., 0] = mask[..., :-1].clone(), 0
            logits[mask.scatter(1, sorted_indices, mask)] = -float('inf')
        
        # ⑤ 采样
        if do_sample:
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        else:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        past_key_values = outputs.past_key_values
        
        marker = " (EOS)" if next_token.item() == eos_token_id else ""
        print(f"  step {step:2d}: → {next_token.item()}{marker}")
        
        if next_token.item() == eos_token_id:
            print("  遇到 EOS,停止生成。")
            break
    
    print(f"\n最终序列({input_ids.shape[1]} tokens): {input_ids[0].tolist()}")
    return input_ids

# 用 FakeModel 演示(vocab=20, 不需要真模型)
torch.manual_seed(42)
fake_model = FakeModel()
fake_input = torch.tensor([[1, 5, 3]])

print("=" * 55)
print("minimind generate 核心逻辑演示(do_sample=True)")
print("=" * 55)
_ = minimind_generate_essence(fake_model, fake_input, max_new_tokens=5, 
                               temperature=0.85, top_p=0.85, top_k=5,
                               eos_token_id=999)  # 不会触发 EOS

&nbsp;

---

## 7.9 实战对比:不同参数生成同一段文本

`generate` 的参数组合决定了输出的「性格」。让我们对比几种典型配置:

In [ ]:
# 对比不同参数组合下的采样行为(用模拟 logits 展示分布差异)
import torch
import numpy as np

def sample_step(logits, temperature=0.85, top_p=0.85, top_k=50, 
                repetition_penalty=1.0, seen_tokens=None, do_sample=True):
    """单步采样,返回选中的 token id。"""
    logits = logits.clone() / temperature
    
    if repetition_penalty != 1.0 and seen_tokens is not None:
        seen = list(set(seen_tokens))
        score = logits[seen]
        logits[seen] = torch.where(score > 0, score / repetition_penalty, score * repetition_penalty)
    
    if top_k > 0 and top_k < logits.shape[0]:
        logits[logits < torch.topk(logits, top_k)[0][..., -1, None]] = -float('inf')
    
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        mask = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1) > top_p
        mask[..., 1:], mask[..., 0] = mask[..., :-1].clone(), 0
        logits[mask.scatter(0, sorted_indices, mask)] = -float('inf')
    
    if do_sample:
        return torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1).item()
    else:
        return torch.argmax(logits, dim=-1).item()

# 模拟:同一组 logits,不同参数各采样 100 次,统计分布
torch.manual_seed(0)
base_logits = torch.randn(50) * 3
base_logits[5] = 5.0
base_logits[12] = 4.0
base_logits[20] = 3.5

configs = [
    ("Greedy (T=0.01)",        dict(temperature=0.01, top_p=1.0, top_k=0, do_sample=False)),
    ("T=0.85, top_p=0.85",     dict(temperature=0.85, top_p=0.85, top_k=0, do_sample=True)),
    ("T=1.0, top_p=0.95",      dict(temperature=1.0,  top_p=0.95, top_k=0, do_sample=True)),
    ("T=1.5 (chaotic)",        dict(temperature=1.5,  top_p=1.0,  top_k=0, do_sample=True)),
    ("Rep penalty=1.3",        dict(temperature=0.85, top_p=0.85, top_k=0, 
                                    repetition_penalty=1.3, seen_tokens=[5,12,20], do_sample=True)),
]

print("=== 同一 logits,不同参数采样 100 次 ===\n")
print(f"{'配置':<22s} {'token5':>7s} {'token12':>8s} {'token20':>8s} {'其他':>6s} {'多样性':>7s}")
print("-" * 62)

for name, params in configs:
    counts = {}
    for _ in range(100):
        tid = sample_step(base_logits.clone(), **params)
        counts[tid] = counts.get(tid, 0) + 1
    
    c5 = counts.get(5, 0)
    c12 = counts.get(12, 0)
    c20 = counts.get(20, 0)
    c_other = sum(v for k, v in counts.items() if k not in (5, 12, 20))
    diversity = len(counts)  # 不同 token 数
    
    print(f"{name:<22s} {c5:>5}% {c12:>6}% {c20:>6}% {c_other:>4}% {diversity:>5}种")

print("\n→ Greedy 100% 选 token5(确定性);高 T 分布更分散;rep penalty 压低已出现 token")
print("→ 多样性 = 采到的不同 token 种数,反映输出的丰富程度")

## Summary and takeaways / 本章总结

| 概念 | 要点 |
|---|---|
| **自回归生成** | 一次一个 token,每步取最后位置 logits,选 token,拼回,重复 |
| **Greedy** | `argmax(logits)` —— 确定性,易重复;T→0 等价 greedy |
| **Temperature** | `logits/T` 再 softmax;T 小→尖锐(确定),T 大→平坦(随机) |
| **Top-k** | 固定保留 k 个;minimind vocab=6400 太小,top_k=50 过于激进 |
| **Top-p** | 累积概率到 p 就停;动态截断,比 top-k 更灵活 |
| **Repetition penalty** | 已出现 token 的 logit 正数除以/负数乘以 penalty;=1 不惩罚 |
| **KV cache** | 缓存历史 K/V,每步只 forward 新 token;O(T²)→O(T) |
| **generate 执行顺序** | temperature → rep penalty → top_k → top_p → multinomial 采样 |
| **EOS 终止** | `finished` 追踪每条序列,全部结束才 break |
| **覆盖 GenerationMixin** | minimind 手写 32 行 generate,不用 HF 的 LogitsProcessor 机制 |

> 生成是 LLM 的「出口」—— logits 是连续的概率空间,生成策略是这个空间到离散 token 序列的**采样映射**。理解 temperature/top-k/top-p/repetition penalty 的叠加效果,你就能精确控制模型的输出风格:从精确的事实回答(T=0.1)到创意写作(T=1.0, top_p=0.95)。

- 精简复习版见 [`./generation.ipynb`](./generation.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

下一章:[第 8 章](../ch08/01_main-chapter-code/README.md)